# Day 9: 单头因果 Attention

QK^T/sqrt(D) 给出位置间分数，Mask 禁止未来，softmax 后权重沿 key 维求和为 1。再与 V 相乘。教学 Mask 的 True=禁止，SDPA bool Mask 的 True=允许，要显式转换。

参考源码：`learning/mini_transformer/model.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY08_14.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 9完整讲义](../docs/DAY09_SINGLE_ATTENTION.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

把每个Query当成“查资料的问题”，每个Key当成“资料的索引”，Value是资料内容。

```text
位置0：只能看资料0
位置1：可以看资料0、1
位置2：可以看资料0、1、2

Query → 与Key点积 → 除sqrt(D_k) → 屏蔽未来 → softmax → 汇集Value
```

假设某一行缩放后的分数为 `[0, ln(3)]`，对应Value为 `[10,20]`：
权重是 `[1/4,3/4]`，输出为17.5。
如果第二个位置属于未来，应把该分数设为负无穷，权重变成 `[1,0]`，输出10。
**把未来分数设成0并不等于屏蔽**：exp(0)=1，仍有权重。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(9, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

去掉 Mask，改变未来输入会影响前缀吗？最后一维 softmax 为什么是 key 维？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：为什么softmax沿最后的Key轴？**
答：一个Query在它可访问的资料间分配权重，不是在不同Query或不同用户之间归一化。

**问：为什么未来分数不能简单填0？**
答：exp(0)不为0；加法Mask常用负无穷来使概率贡献为0。

**问：分数表为什么是T_q×T_kv，而不是D×D？**
答：每个位置对的特征做点积求和，留下Query位置和Key位置两条轴。

**问：只比较生成文字是否相同够吗？**
答：不够，采样和argmax可掩盖logits差异；应比较全部支持输出和数值误差。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY09_SINGLE_ATTENTION.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。